# Linear SVM: Maximum Margin Classification

**Objective:** Understand how a linear Support Vector Machine finds the optimal separating hyperplane by maximizing the margin between classes.

**Key Concepts:**
- Maximum margin classification
- Support vectors (the points that define the boundary)
- Hinge loss vs logistic loss
- Linear SVM vs Logistic Regression

**Package:** `rkhs_kernel_methods` (located in `src/`)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from rkhs_kernel_methods import (
    make_linearly_separable,
    load_all_datasets,
    train_linear_svm,
    train_logistic_regression,
    plot_decision_boundary,
    plot_margin,
    plot_support_vectors,
    evaluate_model,
    # get_support_vectors,
    decision_function,
    compute_margin_width,
    hinge_loss,
    set_style,
    set_random_seed,
)

set_style()
set_random_seed(42)
print("Package loaded successfully.")

## 1. Generate Linearly Separable Data

We generate two Gaussian clusters with enough separation that a straight line can cleanly divide them. Labels are encoded as $-1$ and $+1$ (SVM convention).

In [ ]:
X, y = make_linearly_separable(n_samples=300, cluster_std=0.60, random_state=42)

print(f"X shape: {X.shape}")
print(f"y unique: {np.unique(y)}")
print(f"Class counts: -1: {(y == -1).sum()}, +1: {(y == 1).sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
from rkhs_kernel_methods import plot_dataset
plot_dataset(X, y, title="Linearly Separable Dataset", ax=ax)
plt.tight_layout()
plt.show()

## 2. Train a Linear SVM

A linear SVM solves the optimization problem:

$$\min_{w, b} \frac{1}{2} \|w\|^2 + C \sum_{i=1}^{n} \max\left(0, 1 - y_i (w^\top x_i + b)\right)$$

The **hinge loss** $\max(0, 1 - y_i \hat{y}_i)$ penalizes points that fall inside the margin or are misclassified.

The **margin width** is $2 / \|w\|$. Maximizing the margin is equivalent to minimizing $\|w\|^2$.

In [ ]:
svm_linear = train_linear_svm(X, y, C=1.0, random_state=42)

margin_width = compute_margin_width(svm_linear)
sv_X, sv_y = get_support_vectors(svm_linear)
scores = decision_function(svm_linear, X)
loss = hinge_loss(y, scores)

print(f"Margin width : {margin_width:.3f}")
print(f"Support vectors : {len(sv_X)} out of {len(X)} ({100*len(sv_X)/len(X):.1f}%)")
print(f"Hinge loss : {loss:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

plot_decision_boundary(
    svm_linear, X, y,
    title="Linear SVM: Decision Boundary",
    highlight_sv=True, ax=ax1
)

plot_margin(
    svm_linear, X, y,
    title="Linear SVM: Maximum Margin ($\\frac{2}{\\|w\\|} = " + f"{margin_width:.2f})"
    , ax=ax2
)

plt.tight_layout()
plt.show()

**Observation:** The dashed lines show the margins at $f(x) = \pm 1$. The solid line is the decision boundary $f(x) = 0$. Support vectors (circled in green) lie exactly on the margin boundaries. Only these points matter for the classifier!

## 3. Train Logistic Regression for Comparison

Logistic regression minimizes log-loss instead of hinge loss. It produces a probabilistic output but does **not** maximize the margin in the same way.

In [ ]:
lr = train_logistic_regression(X, y, C=1.0, random_state=42)

svm_metrics = evaluate_model(svm_linear, X, y)
lr_metrics = evaluate_model(lr, X, y)

comparison_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Linear SVM": [svm_metrics["accuracy"], svm_metrics["precision"],
                   svm_metrics["recall"], svm_metrics["f1"]],
    "Logistic Regression": [lr_metrics["accuracy"], lr_metrics["precision"],
                             lr_metrics["recall"], lr_metrics["f1"]],
})
print("\n=== Model Comparison ===")
print(comparison_df.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

plot_decision_boundary(
    svm_linear, X, y,
    title="Linear SVM (Hinge Loss)",
    highlight_sv=True, ax=ax1
)

plot_decision_boundary(
    lr, X, y,
    title="Logistic Regression (Log Loss)",
    highlight_sv=False, ax=ax2
)

plt.tight_layout()
plt.show()

### Key Differences: SVM vs Logistic Regression

| Aspect | Linear SVM | Logistic Regression |
|--------|-----------|-------------------|
| **Loss function** | Hinge loss | Log (cross-entropy) loss |
| **Output** | Signed distance from boundary | Probability $P(y=1 \mid x)$ |
| **Margin** | Explicitly maximized | Not explicitly maximized |
| **Support vectors** | Only boundary points matter | All points influence the boundary |
| **Interpretation** | Geometric (distance-based) | Probabilistic |
| **Sparsity** | Sparse (support vectors only) | Dense (all points + parameters) |

## 4. Understanding the C Parameter (Regularization)

$C$ controls the trade-off between maximizing the margin and minimizing classification error:
- **Small $C$**: Wider margin, allows more misclassifications (high bias, low variance)
- **Large $C$**: Narrower margin, fewer misclassifications tolerated (low bias, high variance)

In [ ]:
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
fig, axes = plt.subplots(1, len(C_values), figsize=(22, 4.5))

for ax, C in zip(axes, C_values):
    model = train_linear_svm(X, y, C=C, random_state=42)
    w = compute_margin_width(model)
    n_sv = len(model.support_vectors_)
    plot_margin(
        model, X, y,
        title=f"C = {C}\nMargin = {w:.2f}, SV = {n_sv}",
        ax=ax
    )

plt.tight_layout()
plt.show()

## 5. Why Margins Matter

The margin principle is at the heart of SVM generalization:

1. **Maximum margin** $\Rightarrow$ classifier is maximally far from both classes $\Rightarrow$ more robust to perturbations.
2. **Only support vectors** determine the boundary $\Rightarrow$ the model is sparse and computationally efficient.
3. **Generalization bound** depends on the margin: larger margin $\Rightarrow$ better generalization (VC theory).
4. **Hinge loss** $\max(0, 1 - y_i f(x_i))$ does not penalize points that are already correctly classified and far from the boundary (unlike log-loss).

This geometric intuition extends naturally to **nonlinear** classification via the **kernel trick**, which we explore in the next notebook.

## Summary

- A linear SVM finds the hyperplane that **maximizes the margin** between classes.
- Support vectors are the data points that lie on the margin boundaries.
- SVM uses **hinge loss**, which focuses on margin violations rather than probability calibration.
- The parameter $C$ controls the regularization strength.
- The margin concept is the foundation for understanding kernelized SVMs.